# ANZSCO Code Finder — Research Notebook

**Purpose**: Explore ANZSCO data, build and evaluate the RAG matching pipeline.

## How to use this notebook
- Run cells top-to-bottom in a fresh session.
- The final cell (`Save Findings`) writes a summary back to `../current_state.md`.
- **Always run `Save Findings` before clearing context** so nothing is lost.

## Session log
| Date | What was done |
|------|---------------|
| 2026-05-07 | Notebook created, environment setup |

---
## 0. Setup & Imports

In [ ]:
import json
import re
import time
import datetime
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from bs4 import BeautifulSoup

sns.set_theme(style='whitegrid')
DATA_RAW = Path('../data/raw')
DATA_PROC = Path('../data/processed')
STATE_FILE = Path('../current_state.md')

print('Setup OK')
print(f'Data raw dir: {DATA_RAW.resolve()}')

---
## 1. ANZSCO Data Scraping

**Source**: Australian Bureau of Statistics (ABS)  
**URL**: https://www.abs.gov.au/statistics/classifications/anzsco-australian-and-new-zealand-standard-classification-occupations/2022  
**Update frequency**: Check ABS release schedule; re-run this section to refresh.

The ANZSCO hierarchy:
- **Major group** (1 digit) — e.g. 2 Professionals
- **Sub-major group** (2 digits) — e.g. 26 ICT Professionals
- **Minor group** (3 digits) — e.g. 261 Business & Systems Analysts
- **Unit group** (4 digits) — e.g. 2611 ICT Business & Systems Analysts
- **Occupation** (6 digits) — e.g. 261111 ICT Business Analyst

In [ ]:
# --- Scraper: ABS ANZSCO 2022 ---
# ABS publishes ANZSCO as structured web pages. We also check for downloadable datasets.

HEADERS = {'User-Agent': 'Mozilla/5.0 (research bot; contact lizanpeter@gmail.com)'}
BASE_URL = 'https://www.abs.gov.au'
ANZSCO_INDEX = 'https://www.abs.gov.au/statistics/classifications/anzsco-australian-and-new-zealand-standard-classification-occupations/2022'

def fetch(url, delay=1.0):
    """Polite fetch with delay."""
    time.sleep(delay)
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r

print(f'Fetching ANZSCO index: {ANZSCO_INDEX}')
resp = fetch(ANZSCO_INDEX)
print(f'Status: {resp.status_code} | Size: {len(resp.content):,} bytes')

In [ ]:
# Parse index page — find links to major groups and downloadable files
soup = BeautifulSoup(resp.text, 'lxml')

# Look for download links (Excel/CSV)
download_links = []
for a in soup.find_all('a', href=True):
    href = a['href']
    if any(ext in href.lower() for ext in ['.xlsx', '.csv', '.xls']):
        download_links.append({'text': a.get_text(strip=True), 'href': href})

print(f'Download links found: {len(download_links)}')
for l in download_links:
    print(f'  {l["text"]:50s} → {l["href"]}')

In [ ]:
# Find all internal links that look like ANZSCO group pages
group_links = []
for a in soup.find_all('a', href=True):
    href = a['href']
    if 'anzsco' in href.lower() and '2022' in href:
        full = BASE_URL + href if href.startswith('/') else href
        group_links.append({'text': a.get_text(strip=True), 'url': full})

# Deduplicate
group_links = list({l['url']: l for l in group_links}.values())
print(f'ANZSCO group page links found: {len(group_links)}')
for l in group_links[:20]:
    print(f'  {l["text"]:60s} → {l["url"]}')

In [ ]:
# Save raw index HTML for audit trail
raw_path = DATA_RAW / 'anzsco_index.html'
raw_path.write_text(resp.text, encoding='utf-8')

# Save metadata
meta = {
    'scraped_at': datetime.datetime.utcnow().isoformat() + 'Z',
    'source_url': ANZSCO_INDEX,
    'download_links': download_links,
    'group_links_count': len(group_links),
}
(DATA_RAW / 'scrape_meta.json').write_text(json.dumps(meta, indent=2))
print(f'Saved raw HTML → {raw_path}')
print(f'Saved metadata → {DATA_RAW / "scrape_meta.json"}')

---
## 2. Data Structure Analysis

Understand what we have: number of codes at each level, description length, skill levels.

In [ ]:
# Load scraped occupations into a DataFrame
# (Run after scraping cells above have populated data)
data_file = DATA_PROC / 'anzsco_occupations.json'

if data_file.exists():
    occupations = pd.read_json(data_file)
    print(f'Loaded {len(occupations):,} occupations')
    print(occupations.head())
    print('\nColumns:', list(occupations.columns))
else:
    print('No processed data yet — run scraping cells first.')

In [ ]:
# Distribution analysis (run after data is loaded)
if 'occupations' in dir() and len(occupations) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Codes per major group
    if 'major_group' in occupations.columns:
        occupations['major_group'].value_counts().plot(kind='bar', ax=axes[0])
        axes[0].set_title('Occupations per Major Group')
        axes[0].set_xlabel('Major Group')

    # Description length distribution
    if 'description' in occupations.columns:
        desc_lens = occupations['description'].str.len().dropna()
        axes[1].hist(desc_lens, bins=30, color='steelblue')
        axes[1].set_title('Description Length (chars)')
        axes[1].set_xlabel('Characters')
        print(f'Description length — median: {desc_lens.median():.0f}, min: {desc_lens.min()}, max: {desc_lens.max()}')

    # Skill level distribution
    if 'skill_level' in occupations.columns:
        occupations['skill_level'].value_counts().plot(kind='bar', ax=axes[2])
        axes[2].set_title('Skill Level Distribution')

    plt.tight_layout()
    plt.savefig('../data/processed/anzsco_distribution.png', dpi=150)
    plt.show()
else:
    print('Load data first.')

---
## 3. RAG Prototype

Embed ANZSCO descriptions → store in vector DB → retrieve top 5 for a CV snippet.

In [ ]:
# Placeholder — will be built once data scraping is complete
# Steps:
# 1. Embed each ANZSCO description with sentence-transformers
# 2. Store in ChromaDB
# 3. At query time: extract CV text → embed → cosine similarity → top 5
# 4. Re-rank with Claude: add keyword overlap + title match boost
# 5. Compute composite match quality score
print('RAG prototype — pending data scraping')

---
## 4. ATS Readability Score

Score CV text quality before matching — helps users understand why a poor CV gets weak results.

In [ ]:
import textstat

def ats_score(text: str) -> dict:
    """Return ATS-style readability metrics for CV text."""
    return {
        'flesch_reading_ease': textstat.flesch_reading_ease(text),
        'flesch_kincaid_grade': textstat.flesch_kincaid_grade(text),
        'word_count': textstat.lexicon_count(text),
        'sentence_count': textstat.sentence_count(text),
        'avg_sentence_length': textstat.avg_sentence_length(text),
    }

# Quick smoke test
sample = """Software Engineer with 5 years experience in Python, Django, and AWS.
Led development of microservices handling 1M daily requests. 
Managed a team of 3 junior developers. Delivered 12 production releases."""

scores = ats_score(sample)
for k, v in scores.items():
    print(f'{k:30s}: {v}')

---
## 5. Acceptance Test Evaluation

10 ground-truth CVs. Pass criteria: correct code in top 5 for ≥8/10.

In [ ]:
# Ground truth — will be populated with real CV snippets
GROUND_TRUTH = [
    {'role': 'Software Engineer',   'expected_code': '261313'},
    {'role': 'Registered Nurse',    'expected_code': '254411'},
    {'role': 'Accountant',          'expected_code': '221111'},
    {'role': 'Chef',                'expected_code': '351311'},
    {'role': 'Civil Engineer',      'expected_code': '233211'},
    {'role': 'Secondary Teacher',   'expected_code': '241411'},
    {'role': 'Electrician',         'expected_code': '341111'},
    {'role': 'Marketing Manager',   'expected_code': '131011'},
    {'role': 'Architect',           'expected_code': '232111'},
    {'role': 'HR Manager',          'expected_code': '132111'},
]

print('Ground truth test cases:')
for i, gt in enumerate(GROUND_TRUTH, 1):
    print(f'  {i:2d}. {gt["role"]:25s} → ANZSCO {gt["expected_code"]}')

In [ ]:
def evaluate_results(results: list[dict], ground_truth: list[dict]) -> dict:
    """Evaluate top-5 results against ground truth."""
    top5_hits, top1_hits = 0, 0
    rows = []
    for gt, res in zip(ground_truth, results):
        predicted_codes = [r['code'] for r in res.get('top5', [])]
        in_top5 = gt['expected_code'] in predicted_codes
        is_top1 = predicted_codes[0] == gt['expected_code'] if predicted_codes else False
        top5_hits += in_top5
        top1_hits += is_top1
        rows.append({'role': gt['role'], 'expected': gt['expected_code'],
                     'in_top5': in_top5, 'is_top1': is_top1,
                     'predicted_top1': predicted_codes[0] if predicted_codes else None})
    n = len(ground_truth)
    return {
        'top5_accuracy': top5_hits / n,
        'top1_accuracy': top1_hits / n,
        'pass': top5_hits / n >= 0.80,
        'stretch': top1_hits / n >= 0.50,
        'detail': pd.DataFrame(rows)
    }

print('Evaluator ready — run after RAG pipeline is built.')

---
## 6. Save Findings to current_state.md

**Run this cell before clearing context.** It appends a timestamped findings summary to `current_state.md`.

In [ ]:
def save_findings(findings: str):
    """Append a timestamped findings block to current_state.md."""
    state = STATE_FILE.read_text(encoding='utf-8')
    timestamp = datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')
    block = f"\n### Findings — {timestamp}\n{findings.strip()}\n"

    # Insert under the ## Data Findings section
    marker = '## Data Findings'
    if marker in state:
        state = state.replace(
            '_Populated as analysis progresses._',
            '_Populated as analysis progresses._' + block
        )
    else:
        state += block

    STATE_FILE.write_text(state, encoding='utf-8')
    print(f'Saved findings to {STATE_FILE.resolve()}')
    print('--- Preview ---')
    print(block)

# --- EDIT THIS before running ---
MY_FINDINGS = """
- TODO: replace with actual findings from this session
- Example: ABS ANZSCO index page returned 200, found N download links
- Example: Total occupation codes scraped: N (N unit groups, N occupations)
"""

save_findings(MY_FINDINGS)